In [ ]:
# ============================================================
# KAGGLE TRAINING: Stream data from multiple batch folders
# ============================================================
#
# SETUP:
# 1. Upload each batch as a separate Kaggle dataset
# 2. Add all batch datasets as inputs to your notebook
# 3. Run this training script
#
# The DataLoader will stream from disk - never loading all data at once
# ============================================================

import os
import numpy as np
import nibabel as nib
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
import random

# ==================== CONFIGURATION ====================
# Paths to your batch datasets (adjust based on your Kaggle input paths)
BATCH_PATHS = [
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-01/batch_01',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-02/batch_02',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-03/batch_03',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-04/batch_04',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-05/batch_05',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-06/batch_06',
    '/kaggle/input/datasets/kazmirfahrier/thesis-batch-07/batch_07',
]

# Class mapping
CLASS_NAMES = ['Left leg movements', 'Right leg movements', 'Forearm movements', 'Upper arm movements']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# Training config
BATCH_SIZE = 8  # Small batch size to fit in memory
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")

# ==================== STREAMING DATASET ====================
class StreamingMRIDataset(Dataset):
    """
    Dataset that loads NIfTI files on-demand (streaming)
    Only keeps file paths in memory, not actual data
    """
    
    def __init__(self, batch_paths, class_names, transform=None):
        """
        Args:
            batch_paths: List of paths to batch directories
            class_names: List of class folder names
            transform: Optional transform to apply
        """
        self.transform = transform
        self.samples = []  # List of (file_path, label) tuples
        
        # Scan all batches and collect file paths
        for batch_path in batch_paths:
            batch_path = Path(batch_path)
            if not batch_path.exists():
                print(f"Warning: {batch_path} not found")
                continue
            
            for class_name in class_names:
                class_idx = CLASS_TO_IDX[class_name]
                class_dir = batch_path / class_name
                
                if not class_dir.exists():
                    continue
                
                for nii_file in class_dir.glob('*.nii.gz'):
                    self.samples.append((str(nii_file), class_idx))
        
        print(f"Total samples found: {len(self.samples)}")
        
        # Count per class
        class_counts = {}
        for _, label in self.samples:
            class_counts[label] = class_counts.get(label, 0) + 1
        
        print("Class distribution:")
        for idx, name in enumerate(class_names):
            print(f"  {name}: {class_counts.get(idx, 0)}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        """Load single sample on-demand"""
        file_path, label = self.samples[idx]
        
        # Load NIfTI file
        img = nib.load(file_path)
        data = img.get_fdata().astype(np.float32)
        
        # Add channel dimension: (100, 100, 100) -> (1, 100, 100, 100)
        data = np.expand_dims(data, axis=0)
        
        # Convert to tensor
        data = torch.from_numpy(data)
        
        if self.transform:
            data = self.transform(data)
        
        return data, label

# ==================== MODEL (from your thesis) ====================
class AttentionBlock3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query = nn.Conv3d(in_channels, in_channels // 8, kernel_size=1)
        self.key = nn.Conv3d(in_channels, in_channels // 8, kernel_size=1)
        self.value = nn.Conv3d(in_channels, in_channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        batch, C, D, H, W = x.size()
        
        q = self.query(x).view(batch, -1, D * H * W).permute(0, 2, 1)
        k = self.key(x).view(batch, -1, D * H * W)
        v = self.value(x).view(batch, -1, D * H * W)
        
        attention = torch.softmax(torch.bmm(q, k), dim=-1)
        out = torch.bmm(v, attention.permute(0, 2, 1))
        out = out.view(batch, C, D, H, W)
        
        return self.gamma * out + x

class CNN3D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        
        self.norm = nn.InstanceNorm3d(1)
        
        self.conv1 = nn.Conv3d(1, 32, kernel_size=7, stride=2, padding=3)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=7, stride=2, padding=3)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=7, stride=2, padding=3)
        
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.attention = AttentionBlock3D(128)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.norm(x)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.pool(x)
        x = self.attention(x)
        
        x = x.view(x.size(0), 1, -1)  # (batch, 1, 128)
        x = self.transformer(x)
        x = x.squeeze(1)  # (batch, 128)
        
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# ==================== TRAINING FUNCTIONS ====================
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training")
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{running_loss/total:.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    return running_loss / len(dataloader), correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, labels in tqdm(dataloader, desc="Validation"):
            data = data.to(device)
            labels = labels.to(device)
            
            outputs = model(data)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(dataloader), correct / total

# ==================== MAIN TRAINING LOOP ====================
def main():
    print("="*60)
    print("STREAMING TRAINING ON FULL 62-SUBJECT DATASET")
    print("="*60)
    
    # Create dataset
    print("\nLoading dataset paths...")
    full_dataset = StreamingMRIDataset(BATCH_PATHS, CLASS_NAMES)
    
    # Split into train/val/test (80/10/10)
    total_samples = len(full_dataset)
    indices = list(range(total_samples))
    random.shuffle(indices)
    
    train_size = int(0.8 * total_samples)
    val_size = int(0.1 * total_samples)
    
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]
    
    print(f"\nSplit: Train={len(train_indices)}, Val={len(val_indices)}, Test={len(test_indices)}")
    
    # Create subset datasets
    train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
    val_dataset = torch.utils.data.Subset(full_dataset, val_indices)
    test_dataset = torch.utils.data.Subset(full_dataset, test_indices)
    
    # Create dataloaders (num_workers=0 for Kaggle compatibility)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Create model
    print("\nCreating model...")
    model = CNN3D(num_classes=NUM_CLASSES).to(DEVICE)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
    
    # Training loop
    print("\nStarting training...")
    best_val_acc = 0.0
    
    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"{'='*60}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
        
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
            print(f"*** New best model saved! Val Acc: {val_acc*100:.2f}% ***")
    
    # Final evaluation on test set
    print("\n" + "="*60)
    print("FINAL EVALUATION ON TEST SET")
    print("="*60)
    
    model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
    test_loss, test_acc = validate(model, test_loader, criterion, DEVICE)
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc*100:.2f}%")
    
    print("\nTraining complete!")

if __name__ == "__main__":
    main()